<a href="https://colab.research.google.com/github/qurbanovkenan77-tech/mgmt467-analytics-portfolio/blob/main/Labs/Unit2_Lab1_PromptPlusExamples_Colab_Kaggle_GCS_BQ_DQ_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MGMT 467 — Prompt-Driven Lab (with Commented Examples)
## Kaggle ➜ Google Cloud Storage ➜ BigQuery ➜ Data Quality (DQ)

**How to use this notebook**
- Each section gives you a **Build Prompt** to paste into Gemini/Vertex AI (or Gemini in Colab).
- Below each prompt, you’ll see a **commented example** of what a good LLM answer might look like.
- **Do not** just uncomment and run. Use the prompt to generate your own code, then compare to the example.
- After every step, run the **Verification Prompt**, and write the **Reflection** in Markdown.

> Goal today: Download the Netflix dataset (Kaggle) → Stage on GCS → Load into BigQuery → Run DQ profiling (missingness, duplicates, outliers, anomaly flags).


### Academic integrity & LLM usage
- Use the prompts here to generate your own code cells.
- Read concept notes and write the reflection answers in your own words.
- Keep credentials out of code. Upload `kaggle.json` when asked.


## Learning objectives
1) Explain **why** we stage data in GCS and load it to BigQuery.  
2) Build an **idempotent**, auditable pipeline.  
3) Diagnose **missingness**, **duplicates**, and **outliers** and justify cleaning choices.  
4) Connect DQ decisions to **business/ML impact**.


## 0) Environment setup — What & Why
Authenticate Colab to Google Cloud so we can use `gcloud`, GCS, and BigQuery. Set **PROJECT_ID** and **REGION** once for consistency (cost/latency).

### Build Prompt (paste to LLM)
You are my cloud TA. Generate a single **Colab code cell** that:
1) Authenticates to Google Cloud in Colab,  
2) Prompts for `PROJECT_ID` via `input()` and sets `REGION="us-central1"` (editable),  
3) Exports `GOOGLE_CLOUD_PROJECT`,  
4) Runs `gcloud config set project $GOOGLE_CLOUD_PROJECT`,  
5) Prints both values. Add 2–3 comments explaining what/why.
End with a comment: `# Done: Auth + Project/Region set`.


In [ ]:
# Authenticate Google Colab with Google Cloud
from google.colab import auth
auth.authenticate_user()

import os

# Prompt user to enter their Google Cloud Project ID
project_id = input("Enter your Google Cloud Project ID: ").strip()

# Set a default region (update if instructed)
region = "us-central1"

# Export the project ID as an environment variable for later use
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id

# Display confirmation
print(f"Project set to: {project_id}")
print(f"Region set to: {region}")

# Configure gcloud CLI to use this project by default
!gcloud config set project $GOOGLE_CLOUD_PROJECT
!gcloud config get-value project

# Explanation:
# 1. Authenticates Colab to access Google Cloud services.
# 2. Prompts user for project ID and sets region for consistency.
# 3. Exports the project variable and configures gcloud CLI.
# Done: Auth + Project/Region set


Enter your Google Cloud Project ID: mgmt-467-471119
Project set to: mgmt-467-471119
Region set to: us-central1
Updated property [core/project].
mgmt-467-471119


In [ ]:
# # EXAMPLE (from LLM) — Auth + Project/Region (commented; write your own cell using the prompt)
# # from google.colab import auth
# # auth.authenticate_user()
# #
# # import os
# # PROJECT_ID = input("Enter your GCP Project ID: ").strip()
# # REGION = "us-central1"  # keep consistent; change if instructed
# # os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
# # print("Project:", PROJECT_ID, "| Region:", REGION)
# #
# # # Set active project for gcloud/BigQuery CLI
# # !gcloud config set project $GOOGLE_CLOUD_PROJECT
# # !gcloud config get-value project
# # # Done: Auth + Project/Region set

### Verification Prompt
Generate a short cell that prints the active project using `gcloud config get-value project` and echoes the `REGION` you set.


In [ ]:
# ==== Verification: project & region are set the way we think ====
import os
from IPython.utils import io

# 1) What does the gcloud CLI think the active project is?
with io.capture_output() as cap:
    # returns a one-line string like: mgmt-467-471119
    pass
active_project = !gcloud config get-value project
active_project = (active_project or ["<none>"])[0].strip()

# 2) What did we export as the env var?
env_project  = os.getenv("GOOGLE_CLOUD_PROJECT", "<not set>")
# Region: prefer the Python var 'region' if it exists, else env var, else '<not set>'
region_value = globals().get("region", os.getenv("REGION", "<not set>"))

print("gcloud active project :", active_project)
print("env GOOGLE_CLOUD_PROJECT:", env_project)
print("REGION                 :", region_value)

# 3) Hard check: CLI vs env must agree
assert active_project == env_project, (
    "Mismatch: gcloud config project != GOOGLE_CLOUD_PROJECT. "
    "Re-run the setup cell to keep them consistent."
)

print("✅ Verification passed: project + region look consistent.")



gcloud active project : mgmt-467-471119
env GOOGLE_CLOUD_PROJECT: mgmt-467-471119
REGION                 : <not set>
✅ Verification passed: project + region look consistent.


**Reflection:** Why do we set `PROJECT_ID` and `REGION` at the top? What can go wrong if we don’t?

- We set PROJECT_ID and a single REGION at the very top to keep every tool in sync (Python client, gcloud CLI, and SQL).

- If we don’t, it’s easy to end up with silent mismatches—e.g., gcloud writing to one project while BigQuery Python writes to another.

- Inconsistent region can cause job failures (location errors), slow performance, or cross-region egress costs.

- Having one source of truth (env var + gcloud config) makes the rest of the notebook repeatable and debuggable.

## 1) Kaggle API — What & Why
Use Kaggle CLI for reproducible downloads. Store `kaggle.json` at `~/.kaggle/kaggle.json` with `0600` permissions to protect secrets.

### Build Prompt
Generate a **single Colab code cell** that:
- Prompts me to upload `kaggle.json`,
- Saves to `~/.kaggle/kaggle.json` with `0600` permissions,
- Prints `kaggle --version`.
Add comments about security and reproducibility.


In [ ]:
# Authenticate Kaggle CLI for dataset access and reproducibility
from google.colab import files
import os

# Step 1: Prompt the user to upload their Kaggle API key (kaggle.json)
print("Upload your kaggle.json file (from Kaggle > Account > Create New API Token):")
user_file = files.upload()

# Step 2: Create the hidden .kaggle directory if it doesn't already exist
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

# Step 3: Save the uploaded kaggle.json securely with restricted access
kaggle_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_path, "wb") as f:
    f.write(user_file[list(user_file.keys())[0]])
os.chmod(kaggle_path, 0o600)

# Step 4: Verify Kaggle installation and setup
!kaggle --version

# Explanation:
# - Prompts user to upload kaggle.json securely.
# - Stores credentials at ~/.kaggle/kaggle.json with 600 permissions (owner read/write only).
# - Enables authenticated Kaggle downloads while keeping API tokens private.
# Done: Kaggle API authenticated and secured.


Upload your kaggle.json file (from Kaggle > Account > Create New API Token):


Saving kaggle.json to kaggle.json
Kaggle API 1.7.4.5


In [ ]:
# # EXAMPLE (from LLM) — Kaggle setup (commented)
# # from google.colab import files
# # print("Upload your kaggle.json (Kaggle > Account > Create New API Token)")
# # uploaded = files.upload()
# #
# # import os
# # os.makedirs('/root/.kaggle', exist_ok=True)
# # with open('/root/.kaggle/kaggle.json', 'wb') as f:
# #     f.write(uploaded[list(uploaded.keys())[0]])
# # os.chmod('/root/.kaggle/kaggle.json', 0o600)  # owner-only
# #
# # !kaggle --version

### Verification Prompt
Generate a one-liner that runs `kaggle --help | head -n 20` to show the CLI is ready.


In [ ]:
# Show CLI is ready
!kaggle --help | head -n 20

# Python script to verify the token file and its permissions
import pathlib, stat

print("\n--- Verification of kaggle.json ---")
path = pathlib.Path('~/.kaggle/kaggle.json').expanduser()
exists = path.exists()
permissions = "missing"
if exists:
  # Get the permissions and format them as an octal string (e.g., '0o600')
  permissions = oct(path.stat().st_mode & 0o777)

print(f"Token file path: {path}")
print(f"File exists:     {exists}")
print(f"Permissions:     {permissions} (expected: 0o600)")

# Assert that the file exists and has the correct permissions
assert exists, "Verification failed: kaggle.json file not found."
assert permissions == '0o600', f"Verification failed: Incorrect permissions. Expected '0o600' but found {permissions}."

print("\n✅ Verification successful: kaggle.json exists with correct permissions.")

usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,config}
              ...

options:
  -h, --help            show this help message and exit
  -v, --version         Print the Kaggle API version
  -W, --no-warn         Disable out-of-date API version warning

commands:
  {competitions,c,datasets,d,kernels,k,models,m,files,f,config}
                        Use one of:
                        competitions {list, files, download, submit, submissions, leaderboard}
                        datasets {list, files, download, create, version, init, metadata, status}
                        kernels {list, files, init, push, pull, output, status}
                        models {instances, get, list, init, create, delete, update}
                        models instances {versions, get, files, init, create, delete, update}
                        models instances versions {init, create, download, delete, files}
                        config {view, set,

**Reflection:** Why require strict `0600` permissions on API tokens? What risks are we avoiding?

- Why 0600? It makes the API key file readable/writable only by the current user. Kaggle’s CLI enforces this; if the file is broader than 0600, it refuses to run to prevent leaks.

- Risks we avoid:

  - Credential leakage to other users on a shared runtime or VM if the file were world- or group-readable.

  - Accidental key exposure via synced folders (e.g., if ~/.kaggle/ were copied into a repo or shared volume).

  - Audit/rule violations—some environments require least-privilege on secrets; 0600 satisfies that.

- If we don’t enforce it: Kaggle CLI may error (e.g., “API credentials file must be readable only by owner”), or worse, your token could be exposed to other users/processes.

## 2) Download & unzip dataset — What & Why
Keep raw files under `/content/data/raw` for predictable paths and auditing.
**Dataset:** `sayeeduddin/netflix-2025user-behavior-dataset-210k-records`

### Build Prompt
Generate a **Colab code cell** that:
- Creates `/content/data/raw`,
- Downloads the dataset to `/content/data` with Kaggle CLI,
- Unzips into `/content/data/raw` (overwrite OK),
- Lists all CSVs with sizes in a neat table.
Include comments describing each step.


In [ ]:
# Download and extract Netflix user behavior dataset for reproducible analysis

# Step 1: Create organized folders for raw data
!mkdir -p /content/data/raw

# Step 2: Use Kaggle CLI to download the dataset into /content/data
# Dataset: sayeeduddin/netflix-2025user-behavior-dataset-210k-records
!kaggle datasets download -d sayeeduddin/netflix-2025user-behavior-dataset-210k-records -p /content/data

# Step 3: Unzip the dataset into /content/data/raw (overwrite if exists)
!unzip -o /content/data/netflix-2025user-behavior-dataset-210k-records.zip -d /content/data/raw

# Step 4: Display all CSV files and their sizes for verification
!ls -lh /content/data/raw/*.csv

# Explanation:
# 1. Creates consistent folder structure under /content/data/raw for auditing and reuse.
# 2. Downloads the dataset securely using authenticated Kaggle CLI.
# 3. Unzips the raw data with overwrite enabled for repeatable runs.
# 4. Lists all CSV files with file sizes to confirm successful extraction.
# Done: Dataset downloaded, extracted, and verified.


Dataset URL: https://www.kaggle.com/datasets/sayeeduddin/netflix-2025user-behavior-dataset-210k-records
License(s): CC0-1.0
  0% 0.00/4.02M [00:00<?, ?B/s]
100% 4.02M/4.02M [00:00<00:00, 493MB/s]
Archive:  /content/data/netflix-2025user-behavior-dataset-210k-records.zip
  inflating: /content/data/raw/README.md  
  inflating: /content/data/raw/movies.csv  
  inflating: /content/data/raw/recommendation_logs.csv  
  inflating: /content/data/raw/reviews.csv  
  inflating: /content/data/raw/search_logs.csv  
  inflating: /content/data/raw/users.csv  
  inflating: /content/data/raw/watch_history.csv  
-rw-r--r-- 1 root root 114K Aug  2 19:36 /content/data/raw/movies.csv
-rw-r--r-- 1 root root 4.5M Aug  2 19:36 /content/data/raw/recommendation_logs.csv
-rw-r--r-- 1 root root 1.8M Aug  2 19:36 /content/data/raw/reviews.csv
-rw-r--r-- 1 root root 2.2M Aug  2 19:36 /content/data/raw/search_logs.csv
-rw-r--r-- 1 root root 1.6M Aug  2 19:36 /content/data/raw/users.csv
-rw-r--r-- 1 root root 8.9M A

In [ ]:
# # EXAMPLE (from LLM) — Download & unzip (commented)
# # !mkdir -p /content/data/raw
# # !kaggle datasets download -d sayeeduddin/netflix-2025user-behavior-dataset-210k-records -p /content/data
# # !unzip -o /content/data/*.zip -d /content/data/raw
# # # List CSV inventory
# # !ls -lh /content/data/raw/*.csv

### Verification Prompt
Generate a snippet that asserts there are exactly **six** CSV files and prints their names.


In [ ]:
# Verify that exactly six CSV files exist in /content/data/raw and print their names
import glob
import os

csv_files = glob.glob("/content/data/raw/*.csv")

# Check count
assert len(csv_files) == 6, f"Expected 6 CSV files, but found {len(csv_files)}"

# Print confirmation and file names
print("Verification passed ✅ — 6 CSV files found:\n")
for file in csv_files:
    size_mb = os.path.getsize(file) / (1024 * 1024)
    print(f"{os.path.basename(file):40s} {size_mb:.2f} MB")


Verification passed ✅ — 6 CSV files found:

watch_history.csv                        8.84 MB
reviews.csv                              1.78 MB
users.csv                                1.53 MB
search_logs.csv                          2.15 MB
recommendation_logs.csv                  4.48 MB
movies.csv                               0.11 MB


**Reflection:** Why is keeping a clean file inventory (names, sizes) useful downstream?

- Keeping a clean inventory of dataset files, with clear names and sizes, ensures data reproducibility, traceability, and consistency in later stages.
When performing cleaning, merging, or modeling, it’s easier to detect missing or corrupted files, automate pipelines, and document data provenance for future audits or collaborators.

## 3) Create GCS bucket & upload — What & Why
Stage in GCS → consistent, versionable source for BigQuery loads. Bucket names must be **globally unique**.

### Build Prompt
Generate a **Colab code cell** that:
- Creates a unique bucket in `${REGION}` (random suffix),
- Saves name to `BUCKET_NAME` env var,
- Uploads all CSVs to `gs://$BUCKET_NAME/netflix/`,
- Prints the bucket name and explains staging benefits.


In [ ]:
# === Create a unique GCS bucket and upload Netflix dataset for BigQuery staging ===
import os, uuid, re

# 1) Define region and bucket name
region = "US"  # multi-region works broadly; change if your course requires a specific region
bucket_name = f"kanan-netflix-{uuid.uuid4().hex[:10]}".lower()

# 2) Expose values to both Python and shell
os.environ["BUCKET_NAME"] = bucket_name
os.environ["REGION"] = region

# 3) Validate bucket name format (GCS rules)
assert re.match(r'^[a-z0-9][a-z0-9\-]{1,61}[a-z0-9]$', bucket_name), f"Invalid bucket name: {bucket_name}"
print(f"Creating bucket: {bucket_name}  |  region: {region}")

# 4) Create bucket (NOTE: use --location=$REGION or --location={region}; both are fine now)
!gcloud storage buckets create gs://$BUCKET_NAME --location=$REGION --quiet

# 5) Upload CSVs and verify
!gcloud storage cp /content/data/raw/*.csv gs://$BUCKET_NAME/netflix/
!gcloud storage ls -L gs://$BUCKET_NAME/netflix/



Creating bucket: kanan-netflix-d267b603b8  |  region: US
Creating gs://kanan-netflix-d267b603b8/...
Copying file:///content/data/raw/movies.csv to gs://kanan-netflix-d267b603b8/netflix/movies.csv
Copying file:///content/data/raw/recommendation_logs.csv to gs://kanan-netflix-d267b603b8/netflix/recommendation_logs.csv
Copying file:///content/data/raw/reviews.csv to gs://kanan-netflix-d267b603b8/netflix/reviews.csv
Copying file:///content/data/raw/search_logs.csv to gs://kanan-netflix-d267b603b8/netflix/search_logs.csv
Copying file:///content/data/raw/users.csv to gs://kanan-netflix-d267b603b8/netflix/users.csv
Copying file:///content/data/raw/watch_history.csv to gs://kanan-netflix-d267b603b8/netflix/watch_history.csv

Average throughput: 60.9MiB/s
gs://kanan-netflix-d267b603b8/netflix/movies.csv:
  Creation Time:               2025-10-21T19:45:32Z
  Update Time:                 2025-10-21T19:45:32Z
  Storage Class Update Time:   2025-10-21T19:45:32Z
  Storage Class:               STANDA

In [ ]:
# # EXAMPLE (from LLM) — GCS staging (commented)
# # import uuid, os
# # bucket_name = f"mgmt467-netflix-{uuid.uuid4().hex[:8]}"
# # os.environ["BUCKET_NAME"] = bucket_name
# # !gcloud storage buckets create gs://$BUCKET_NAME --location=$REGION
# # !gcloud storage cp /content/data/raw/* gs://$BUCKET_NAME/netflix/
# # print("Bucket:", bucket_name)
# # # Verify contents
# # !gcloud storage ls gs://$BUCKET_NAME/netflix/

### Verification Prompt
Generate a snippet that lists the `netflix/` prefix and shows object sizes.


In [ ]:
import subprocess

result = subprocess.run(["gcloud", "storage", "ls", f"gs://{bucket_name}/netflix/"], capture_output=True, text=True)
files = [f for f in result.stdout.strip().split("\n") if f.endswith(".csv")]
assert len(files) > 0, "No CSV files found in the GCS bucket."
print(f"\nVerification passed ✅ — {len(files)} CSV files successfully uploaded to gs://{bucket_name}/netflix/\n")



Verification passed ✅ — 6 CSV files successfully uploaded to gs://kanan-netflix-d267b603b8/netflix/



**Reflection:** Name two benefits of staging in GCS vs loading directly from local Colab.

- Staging data in Google Cloud Storage (GCS) ensures durability, version control, and collaboration.
Unlike Colab’s temporary local storage, GCS persists across sessions and integrates directly with BigQuery.
This approach enables scalable, reproducible analytics and team-wide data access for future projects.



## 4) BigQuery dataset & loads — What & Why
Create dataset `netflix` and load six CSVs with **autodetect** for speed (we’ll enforce schemas later).

### Build Prompt (two cells)
**Cell A:** Create (idempotently) dataset `netflix` in US multi-region; if it exists, print a friendly message.  
**Cell B:** Load tables from `gs://$BUCKET_NAME/netflix/`:
`users, movies, watch_history, recommendation_logs, search_logs, reviews`
with `--skip_leading_rows=1 --autodetect --source_format=CSV`.
Finish with row-count queries for each table.


In [ ]:
# Create the BigQuery dataset if it doesn't exist.
DATASET = "netflix"

# The '|| echo...' part handles the case where the dataset already exists, preventing an error.
!bq --location=US mk -d --description "MGMT467 Netflix dataset" {DATASET} || echo "Dataset '{DATASET}' may already exist."

print(f"BigQuery dataset '{DATASET}' is ready.")

BigQuery error in mk operation: Dataset 'mgmt-467-471119:netflix' already
exists.
Dataset 'netflix' may already exist.
BigQuery dataset 'netflix' is ready.


In [ ]:
# Load all 6 tables from GCS into the 'netflix' BigQuery dataset.
import os

# The dataset name must be defined for the bq commands below.
DATASET = "netflix"
BUCKET_NAME = os.environ.get("BUCKET_NAME")
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")

# A dictionary mapping the desired table name to the source CSV filename.
tables = {
  "users": "users.csv",
  "movies": "movies.csv",
  "watch_history": "watch_history.csv",
  "recommendation_logs": "recommendation_logs.csv",
  "search_logs": "search_logs.csv",
  "reviews": "reviews.csv",
}

# Loop through the dictionary to load each table.
# The f-string formatting correctly inserts Python variables into the shell command.
for tbl, fname in tables.items():
  src_uri = f"gs://{BUCKET_NAME}/netflix/{fname}"
  print(f"Loading table '{tbl}' from '{src_uri}'...")
  !bq load --skip_leading_rows=1 --autodetect --source_format=CSV {DATASET}.{tbl} {src_uri}

# Verify by listing row counts for each table loaded.
print("\n--- Verifying Row Counts ---")
for tbl in tables.keys():
    query = f"SELECT '{tbl}' AS table_name, COUNT(*) AS n FROM `{PROJECT_ID}.{DATASET}.{tbl}`"
    !bq query --nouse_legacy_sql "{query}"

Loading table 'users' from 'gs://kanan-netflix-d267b603b8/netflix/users.csv'...
Waiting on bqjob_r62603b3488f71425_0000019a084e4a37_1 ... (1s) Current status: DONE   
Loading table 'movies' from 'gs://kanan-netflix-d267b603b8/netflix/movies.csv'...
Waiting on bqjob_r452e6209e68c1a03_0000019a084e5d11_1 ... (1s) Current status: DONE   
Loading table 'watch_history' from 'gs://kanan-netflix-d267b603b8/netflix/watch_history.csv'...
Waiting on bqjob_r5c3e34966ceaf540_0000019a084e6f17_1 ... (3s) Current status: DONE   
Loading table 'recommendation_logs' from 'gs://kanan-netflix-d267b603b8/netflix/recommendation_logs.csv'...
Waiting on bqjob_r5050b31f18d334a5_0000019a084e8a46_1 ... (2s) Current status: DONE   
Loading table 'search_logs' from 'gs://kanan-netflix-d267b603b8/netflix/search_logs.csv'...
Waiting on bqjob_rcaccc8e4bcaacb3_0000019a084ea15a_1 ... (2s) Current status: DONE   
Loading table 'reviews' from 'gs://kanan-netflix-d267b603b8/netflix/reviews.csv'...
Waiting on bqjob_r584931

### Verification Prompt
Generate a single query that returns `table_name, row_count` for all six tables in `${GOOGLE_CLOUD_PROJECT}.netflix`.


In [ ]:
# The `Unrecognized name: row_count` error is persistent and unusual, suggesting a deeper issue with
# querying INFORMATION_SCHEMA in this specific context.

# As a robust workaround, we can construct a single query that explicitly counts rows
# from each table and combines them with UNION ALL. This achieves the same goal.

import os
from google.cloud import bigquery

project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

# This query is more explicit and bypasses the problematic INFORMATION_SCHEMA view.
sql_query = f"""
    SELECT 'movies' as table_name, COUNT(*) as row_count FROM `{project_id}.netflix.movies`
    UNION ALL
    SELECT 'users' as table_name, COUNT(*) as row_count FROM `{project_id}.netflix.users`
    UNION ALL
    SELECT 'watch_history' as table_name, COUNT(*) as row_count FROM `{project_id}.netflix.watch_history`
    UNION ALL
    SELECT 'recommendation_logs' as table_name, COUNT(*) as row_count FROM `{project_id}.netflix.recommendation_logs`
    UNION ALL
    SELECT 'search_logs' as table_name, COUNT(*) as row_count FROM `{project_id}.netflix.search_logs`
    UNION ALL
    SELECT 'reviews' as table_name, COUNT(*) as row_count FROM `{project_id}.netflix.reviews`
"""

client = bigquery.Client(project=project_id)
results_df = client.query(sql_query).to_dataframe()

print("Verification successful! Row counts for each table:")
results_df

Verification successful! Row counts for each table:


,table_name,row_count
0,movies,2080
1,reviews,30900
2,recommendation_logs,104000
3,search_logs,53000
4,users,20600
5,watch_history,210000


**Reflection:** When is `autodetect` acceptable? When should you enforce explicit schemas and why?

- autodetect is acceptable during early exploration or quick prototyping when data quality is clean and consistent—such as loading well-formatted CSVs for temporary analysis. However, for production pipelines or critical datasets, it’s better to enforce explicit schemas to prevent schema drift, incorrect data types, and silent load errors. Explicit schemas also improve data reliability, documentation, and downstream performance by making table structures predictable and stable.

## 5) Data Quality (DQ) — Concepts we care about
- **Missingness** (MCAR/MAR/MNAR). Impute vs drop. Add `is_missing_*` indicators.
- **Duplicates** (exact vs near). Double-counted engagement corrupts labels & KPIs.
- **Outliers** (IQR). Winsorize/cap vs robust models. Always **flag** and explain.
- **Reproducibility**. Prefer `CREATE OR REPLACE` and deterministic keys.


### 5.1 Missingness (users) — What & Why
Measure % missing and check if missingness depends on another variable (MAR) → potential bias & instability.

### Build Prompt
Generate **two BigQuery SQL cells**:
1) Total rows and % missing in `region`, `plan_tier`, `age_band` from `users`.
2) `% plan_tier missing by region` ordered descending. Add comments on MAR.


In [ ]:
%%bigquery
-- Missingness snapshot for country and subscription_plan
-- The original query failed because 'region', 'plan_tier', and 'age_band' columns do not exist.
-- This version uses the correct column names 'country' and 'subscription_plan' from the table.
WITH counts AS (
  SELECT
    COUNT(*) AS n,
    COUNTIF(country IS NULL) AS m_country,
    COUNTIF(subscription_plan IS NULL) AS m_plan
  FROM `netflix.users`
)
SELECT
  n,
  ROUND(100 * SAFE_DIVIDE(m_country, n), 2) AS pct_missing_country,
  ROUND(100 * SAFE_DIVIDE(m_plan, n), 2) AS pct_missing_plan
FROM counts;

Query is running:   0%|          |

Downloading:   0%|          |

,n,pct_missing_country,pct_missing_plan
0,20600,0.0,0.0


In [ ]:
%%bigquery
-- Does subscription_plan missingness depend on country? (possible MAR)
SELECT
  COALESCE(country, 'Unknown') AS country,
  COUNT(*) AS n,
  ROUND(100 * SAFE_DIVIDE(COUNTIF(subscription_plan IS NULL), COUNT(*)), 2)
    AS pct_plan_missing
FROM `netflix.users`
GROUP BY country
ORDER BY pct_plan_missing DESC, n DESC;


Query is running:   0%|          |

Downloading:   0%|          |

,country,n,pct_plan_missing
0,USA,14408,0.0
1,Canada,6192,0.0


### Verification Prompt
Generate a query that prints the three missingness percentages from (1), rounded to two decimals.


In [ ]:
%%bigquery
SELECT
  ROUND(100 * SAFE_DIVIDE(COUNTIF(country IS NULL), COUNT(*)), 2)
    AS pct_missing_country,
  ROUND(100 * SAFE_DIVIDE(COUNTIF(subscription_plan IS NULL), COUNT(*)), 2)
    AS pct_missing_plan
FROM `netflix.users`;

Query is running:   0%|          |

Downloading:   0%|          |

,pct_missing_country,pct_missing_plan
0,0.0,0.0


**Reflection:** Which columns are most missing? Hypothesize MCAR/MAR/MNAR and why.

- The results show 0% missingness for both country and subscription_plan, which means there is no data loss risk from missing values in this dataset. Since nothing is missing, there is no evidence of MCAR, MAR, or MNAR patterns at this stage—missingness is simply not present. However, it’s still good practice to add is_missing_* flags for reproducibility and to protect against future data changes if missing values appear later.

### 5.2 Duplicates (watch_history) — What & Why
Find exact duplicate interaction records and keep **one best** per group (deterministic policy).

### Build Prompt
Generate **two BigQuery SQL cells**:
1) Report duplicate groups on `(user_id, movie_id, event_ts, device_type)` with counts (top 20).
2) Create table `watch_history_dedup` that keeps one row per group (prefer higher `progress_ratio`, then `minutes_watched`). Add comments.


In [ ]:
%%bigquery
-- Duplicate groups by natural keys (using watch_date instead of event_ts)
SELECT
  user_id,
  movie_id,
  watch_date,
  device_type,
  COUNT(*) AS dup_count
FROM `netflix.watch_history`
GROUP BY user_id, movie_id, watch_date, device_type
HAVING dup_count > 1
ORDER BY dup_count DESC
LIMIT 20;


Query is running:   0%|          |

Downloading:   0%|          |

,user_id,movie_id,watch_date,device_type,dup_count
0,user_00391,movie_0893,2024-08-26,Laptop,8
1,user_03310,movie_0640,2024-09-08,Smart TV,8
2,user_06417,movie_0590,2024-01-15,Laptop,6
3,user_05404,movie_0691,2025-01-01,Mobile,6
4,user_00249,movie_0203,2024-08-31,Laptop,6
5,user_02369,movie_0673,2024-07-04,Mobile,6
6,user_09972,movie_0536,2025-07-16,Laptop,6
7,user_04657,movie_0857,2025-03-04,Mobile,6
8,user_06535,movie_0890,2025-05-23,Desktop,6
9,user_03176,movie_0534,2024-01-06,Laptop,6


In [ ]:
%%bigquery
-- Keep one best row per group: higher progress_percentage, then watch_duration_minutes
CREATE OR REPLACE TABLE `netflix.watch_history_dedup` AS
SELECT * EXCEPT(rk)
FROM (
  SELECT
    h.*,
    ROW_NUMBER() OVER (
      PARTITION BY user_id, movie_id, watch_date, device_type
      ORDER BY
        COALESCE(progress_percentage, -1) DESC,
        COALESCE(watch_duration_minutes, -1) DESC,
        COALESCE(session_id, '') DESC   -- stable tiebreaker
    ) AS rk
  FROM `netflix.watch_history` h
)
WHERE rk = 1;



Query is running:   0%|          |

""


### Verification Prompt
Generate a before/after count query comparing raw vs `watch_history_dedup`.


In [ ]:
%%bigquery
-- Verify effect of dedup step
WITH raw  AS (SELECT * FROM `netflix.watch_history`),
     dedup AS (SELECT * FROM `netflix.watch_history_dedup`),
     dupe_groups AS (
       SELECT user_id, movie_id, watch_date, device_type, COUNT(*) c
       FROM raw
       GROUP BY 1,2,3,4
       HAVING c > 1
     )
SELECT
  (SELECT COUNT(*) FROM raw)                       AS raw_count,
  (SELECT COUNT(*) FROM dedup)                     AS dedup_count,
  (SELECT COUNT(*) FROM dupe_groups)               AS remaining_dupe_groups,   -- should be 0
  (SELECT SUM(watch_duration_minutes) FROM raw)    AS raw_minutes,
  (SELECT SUM(watch_duration_minutes) FROM dedup)  AS dedup_minutes,
  (SELECT SUM(progress_percentage) FROM raw)       AS raw_progress_sum,
  (SELECT SUM(progress_percentage) FROM dedup)     AS dedup_progress_sum;


Query is running:   0%|          |

Downloading:   0%|          |

,raw_count,dedup_count,remaining_dupe_groups,raw_minutes,dedup_minutes,raw_progress_sum,dedup_progress_sum
0,210000,100000,100000,12175136.8,5796742.4,9642913.6,4592334.4


**Reflection:** Why do duplicates arise (natural vs system-generated)? How do they corrupt labels and KPIs?

- Duplicates likely come from system retries/late writes or multiple pings within the same session. If left in, they inflate engagement KPIs (minutes watched, progress, CTR) and corrupt labels for modeling (e.g., “completed watch” counted twice). Our deterministic rule—pick the row with higher progress_percentage, then watch_duration_minutes (with a stable tiebreaker)—keeps the single most informative record per event and makes the pipeline reproducible.

### 5.3 Outliers (minutes_watched) — What & Why
Estimate extreme values via IQR; report % outliers; **winsorize** to P01/P99 for robustness while also **flagging** extremes.

### Build Prompt
Generate **two BigQuery SQL cells**:
1) Compute IQR bounds for `minutes_watched` on `watch_history_dedup` and report % outliers.
2) Create `watch_history_robust` with `minutes_watched_capped` capped at P01/P99; return quantile summaries before/after.


In [ ]:
%%bigquery
-- IQR bounds and outlier rate for watch_duration_minutes
WITH dist AS (
  SELECT
    APPROX_QUANTILES(watch_duration_minutes, 4)[OFFSET(1)] AS q1,
    APPROX_QUANTILES(watch_duration_minutes, 4)[OFFSET(3)] AS q3
  FROM `netflix.watch_history_dedup`
),
bounds AS (
  SELECT
    q1,
    q3,
    (q3 - q1) AS iqr,
    q1 - 1.5 * (q3 - q1) AS lo,
    q3 + 1.5 * (q3 - q1) AS hi
  FROM dist
)
SELECT
  ANY_VALUE(b.q1) AS q1,
  ANY_VALUE(b.q3) AS q3,
  ANY_VALUE(b.iqr) AS iqr,
  ANY_VALUE(b.lo) AS lo,
  ANY_VALUE(b.hi) AS hi,
  COUNTIF(h.watch_duration_minutes < b.lo OR h.watch_duration_minutes > b.hi) AS outliers,
  COUNT(*) AS total,
  ROUND(
    100 * SAFE_DIVIDE(
      COUNTIF(h.watch_duration_minutes < b.lo OR h.watch_duration_minutes > b.hi),
      COUNT(*)
    ),
    2
  ) AS pct_outliers
FROM `netflix.watch_history_dedup` h
CROSS JOIN bounds b;



Query is running:   0%|          |

Downloading:   0%|          |

,q1,q3,iqr,lo,hi,outliers,total,pct_outliers
0,28.9,82.5,53.6,-51.5,162.9,3482,100000,3.48


In [ ]:
%%bigquery
-- Winsorize minutes to P01/P99 and save robust table
CREATE OR REPLACE TABLE `netflix.watch_history_robust` AS
WITH q AS (
  SELECT
    APPROX_QUANTILES(watch_duration_minutes, 100)[OFFSET(1)]  AS p01,
    APPROX_QUANTILES(watch_duration_minutes, 100)[OFFSET(99)] AS p99
  FROM `netflix.watch_history_dedup`
)
SELECT
  h.*,
  GREATEST(q.p01, LEAST(q.p99, h.watch_duration_minutes)) AS watch_duration_minutes_capped
FROM `netflix.watch_history_dedup` h, q;


Query is running:   0%|          |

""


### Verification Prompt
Generate a query that shows min/median/max before vs after capping.


In [ ]:
%%bigquery
WITH before AS (
  SELECT
    'before' AS which,
    MIN(watch_duration_minutes) AS min_val,
    APPROX_QUANTILES(watch_duration_minutes, 3)[OFFSET(1)] AS median_val,
    MAX(watch_duration_minutes) AS max_val
  FROM `netflix.watch_history_dedup`
),
after AS (
  SELECT
    'after' AS which,
    MIN(watch_duration_minutes_capped) AS min_val,
    APPROX_QUANTILES(watch_duration_minutes_capped, 3)[OFFSET(1)] AS median_val,
    MAX(watch_duration_minutes_capped) AS max_val
  FROM `netflix.watch_history_robust`
)
SELECT * FROM before
UNION ALL
SELECT * FROM after;


Query is running:   0%|          |

Downloading:   0%|          |

,which,min_val,median_val,max_val
0,after,4.4,35.7,366.0
1,before,0.2,35.7,799.3


**Reflection:** When might capping be harmful? Name a model type less sensitive to outliers and why.

- Capping (winsorizing) reduces the impact of extreme values that may come from logging errors or rare anomalies, making the data more stable for modeling. However, capping can be harmful when extreme values carry real business signal—for example, users who binge-watch for very long sessions may be valuable power users and should not be suppressed. Tree-based models like Random Forests or Gradient Boosted Trees are less sensitive to outliers because they split on rank/order rather than being influenced by extreme numeric values like linear regression would.

### 5.4 Business anomaly flags — What & Why
Human-readable flags help both product decisioning and ML features (e.g., binge behavior).

### Build Prompt
Generate **three BigQuery SQL cells** (adjust if columns differ):
1) In `watch_history_robust`, compute and summarize `flag_binge` for sessions > 8 hours.
2) In `users`, compute and summarize `flag_age_extreme` if age can be parsed from `age_band` (<10 or >100).
3) In `movies`, compute and summarize `flag_duration_anomaly` where `duration_min` < 15 or > 480 (if exists).
Each cell should output count and percentage and include 1–2 comments.


In [ ]:
%%bigquery
-- Sessions longer than 8 hours (480 minutes)
SELECT
  COUNTIF(watch_duration_minutes_capped > 8*60) AS sessions_over_8h,
  COUNT(*) AS total,
  ROUND(100 * SAFE_DIVIDE(
    COUNTIF(watch_duration_minutes_capped > 8*60), COUNT(*)
  ), 2) AS pct
FROM `netflix.watch_history_robust`;


Query is running:   0%|          |

Downloading:   0%|          |

,sessions_over_8h,total,pct
0,0,100000,0.0


In [ ]:
%%bigquery
SELECT
  COUNTIF(age < 10 OR age > 100) AS extreme_age_rows,
  COUNT(*) AS total,
  ROUND(100 * SAFE_DIVIDE(
    COUNTIF(age < 10 OR age > 100), COUNT(*)
  ), 2) AS pct
FROM `netflix.users`;

Query is running:   0%|          |

Downloading:   0%|          |

,extreme_age_rows,total,pct
0,358,20600,1.74


In [ ]:
%%bigquery
SELECT
  COUNTIF(duration_minutes < 15 OR duration_minutes > 480) AS titles_anomalous_duration,
  COUNT(*) AS total,
  ROUND(100 * SAFE_DIVIDE(
    COUNTIF(duration_minutes < 15 OR duration_minutes > 480), COUNT(*)
  ), 2) AS pct
FROM `netflix.movies`;

Query is running:   0%|          |

Downloading:   0%|          |

,titles_anomalous_duration,total,pct
0,46,2080,2.21


### Verification Prompt
Generate a single compact summary query that returns two columns per flag: `flag_name, pct_of_rows`.


In [ ]:
%%bigquery
WITH binge AS (
  SELECT 'flag_binge' AS flag_name,
         ROUND(100 * SAFE_DIVIDE(
           COUNTIF(watch_duration_minutes_capped > 8*60), COUNT(*)
         ), 2) AS pct_of_rows
  FROM `netflix.watch_history_robust`
),
age AS (
  SELECT 'flag_age_extreme' AS flag_name,
         ROUND(100 * SAFE_DIVIDE(
           COUNTIF(age < 10 OR age > 100), COUNT(*)
         ), 2) AS pct_of_rows
  FROM `netflix.users`
),
dur AS (
  SELECT 'flag_duration_anomaly' AS flag_name,
         ROUND(100 * SAFE_DIVIDE(
           COUNTIF(duration_minutes < 15 OR duration_minutes > 480), COUNT(*)
         ), 2) AS pct_of_rows
  FROM `netflix.movies`
)
SELECT * FROM binge
UNION ALL
SELECT * FROM age
UNION ALL
SELECT * FROM dur;


Query is running:   0%|          |

Downloading:   0%|          |

,flag_name,pct_of_rows
0,flag_binge,0.00
1,flag_age_extreme,1.74
2,flag_duration_anomaly,2.21


**Reflection:** Which anomaly flag is most common? Which would you keep as a feature and why?

- From the summary, the most common flag is flag_duration_anomaly (2.21%), followed by flag_age_extreme (1.74%), while flag_binge is 0.00% (expected after P01/P99 capping). I’d keep flag_duration_anomaly as a modeling feature—it can predict low-quality titles or atypical consumption and is frequent enough to learn from. I’d treat flag_age_extreme mainly as a data-quality guardrail (possible entry errors), optionally one-hot it if you suspect true edge demographics matter. If you need a binge signal, compute it pre-cap or as a rolling user-level metric (e.g., # sessions >3h in last 7 days), since winsorization zeroed out >8h sessions.

## 6) Save & submit — What & Why
Reproducibility: save artifacts and document decisions so others can rerun and audit.

### Build Prompt
Generate a checklist (Markdown) students can paste at the end:
- Save this notebook to the team Drive.
- Export a `.sql` file with your DQ queries and save to repo.
- Push notebook + SQL to the **team GitHub** with a descriptive commit.
- Add a README with your `PROJECT_ID`, `REGION`, bucket, dataset, and today’s row counts.


## Grading rubric (quick)
- Profiling completeness (30)  
- Cleaning policy correctness & reproducibility (40)  
- Reflection/insight (20)  
- Hygiene (naming, verification, idempotence) (10)
